In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch import manual_seed
%matplotlib inline

In [2]:
words  = open('names.txt', 'r').read().split()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [3]:
chars = sorted(list(set(''.join(words))))
stoi  = {s : i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i : s for s, i in stoi.items()}
print(itos)


{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [4]:
#dataset building

block_size = 3
X , Y = [] , []

for w in words:
    #print(w)
    context = [0]*block_size
    for ch in w+'.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        #print('.'.join(itos[i] for i in context), '------->' , itos[ix])
        context = context[1:] + [ix]

X = torch.tensor(X)
Y = torch.tensor(Y)


In [5]:
def build_dataset(words):
    block_size = 3
    X , Y = [] , []

    for w in words:
        #print(w)
        context = [0]*block_size
        for ch in w+'.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            #print('.'.join(itos[i] for i in context), '------->' , itos[ix])
            context = context[1:] + [ix]

    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape , Y.shape)
    return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr , Ytr = build_dataset(words[:n1])
Xdev ,Ydev = build_dataset(words[n1:n2] )
Xte , Yte = build_dataset(words[n2:])

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [6]:
C = torch.randn((27,2))

In [7]:
emb = C[X]
emb.shape

torch.Size([228146, 3, 2])

In [8]:
W1 = torch.randn((6,100) , requires_grad= True)
b1 = torch.randn((100), requires_grad= True)

In [9]:
 h = torch.tanh(emb.view(-1,6) @ W1 + b1)

In [10]:
W2 = torch.randn((100, 27) , requires_grad= True)
b2 = torch.randn((27), requires_grad= True)

In [11]:
logits =  h @ W2 + b2

In [12]:
counts = logits.exp()

In [13]:
probs  = counts/ counts.sum(1 , keepdim= True)

In [14]:
#---------------------------------cleaner----------------------------------------------#

In [15]:
Xtr.shape ,Ytr.shape

(torch.Size([182625, 3]), torch.Size([182625]))

In [16]:
g = torch.Generator().manual_seed(2147483647)
C =  torch.randn((27,10), requires_grad=True, generator= g)
W1 = torch.randn((30, 200) , requires_grad= True, generator = g)
b1 = torch.randn((200), requires_grad= True, generator = g)
W2 = torch.randn((200,27) , requires_grad= True, generator = g)
b2 = torch.randn((27), requires_grad= True, generator = g)
parameters = [C, W1, b1, W2, b2]

In [17]:
sum(p.nelement() for p in parameters)

11897

In [18]:
lre = torch.linspace(-3 , 0, 1000)
lrs = 10 **lre

In [37]:
lri = []
lossi = []
stepi = []

for i in range(100000):

    ix = torch.randint(0 ,Xtr.shape[0], (32,))


    emb = C[Xtr[ix]]
    h = torch.tanh(emb.view(-1,30) @ W1 + b1)
    logits = h @ W2 + b2
    #counts = logits.exp()
    #probs = counts/counts.sum(1 , keepdims=True)
    #loss = -probs[torch.arange(32) , Y].log().mean()
    loss = F.cross_entropy(logits , Ytr[ix])
    print(loss.item())
    lr = 0.0004
    for p in parameters:
        p.grad = None
    loss.backward()

    for p in parameters:
        p.data += -lr * p.grad

    #lri.append(lre[i])
    stepi.append(i)
    lossi.append(loss.log10().item())

2.347578763961792
2.2372474670410156
2.145054340362549
2.4330942630767822
1.690422773361206
1.8337748050689697
1.8018242120742798
1.8625495433807373
2.2884881496429443
2.4773736000061035
2.3321855068206787
2.1526408195495605
2.2310211658477783
2.4674365520477295
1.7743228673934937
2.394094228744507
2.2221620082855225
2.4988315105438232
2.3189144134521484
2.2108685970306396
2.001478910446167
1.8514881134033203
2.0849099159240723
2.4055092334747314
2.2402944564819336
2.2993290424346924
2.078916311264038
2.1207706928253174
2.138702630996704
2.0660784244537354
2.0335772037506104
2.022829532623291
2.0736002922058105
2.324950933456421
1.8678573369979858
2.4262986183166504
1.9615041017532349
2.3341400623321533
2.08535099029541
1.7590409517288208
2.4298923015594482
1.7422542572021484
2.4312751293182373
2.511165142059326
2.026592254638672
1.7674747705459595
2.242694139480591
2.0053415298461914
1.9819122552871704
2.3201138973236084
2.311110019683838
2.34432053565979
2.0264947414398193
2.08279252

In [ ]:
plt.plot(stepi , lossi)

In [38]:
emb = C[Xdev]
h = torch.tanh(emb.view(-1,30) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits , Ydev)
print(loss.item())

2.1680736541748047


In [39]:
emb = C[Xte]
h = torch.tanh(emb.view(-1,30) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits , Yte)
print(loss.item())

2.1713244915008545


In [40]:
for _ in range(20):

    out = []
    context = [0]*block_size
    while True:
        emb  = C[torch.tensor([context])]
        h = torch.tanh(emb.view(1,-1) @ W1 + b1)
        logits = h @ W2 + b2
        probs = F.softmax(logits , dim=1)
        ix = torch.multinomial(probs , num_samples=1, generator= g).item()
        context = context[:1] + [ix]
        out.append(ix)
        if ix == 0:
            break

        print(''.join(itos[i] for i in out))

s


RuntimeError: mat1 and mat2 shapes cannot be multiplied (1x20 and 30x200)